# 02 · Naive baseline

Last-observed value broadcast across the 28-day horizon — the cheapest 
possible baseline. Score is the bottom-level WRMSSE; for the 12-level 
competition score, drop the predictions in `artifacts/cv_naive.parquet` 
and run `06_score.ipynb`.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd

from m5.config import SETTINGS, set_global_seed
from m5.data import split_train_horizon
from m5.evaluation import compute_components, make_submission, wrmsse_for_models
from m5.plots import configure_style

configure_style()
set_global_seed()

## Build the naive forecast

In [ ]:
df = pd.read_parquet(SETTINGS.processed_dir / 'long.parquet')
train, holdout = split_train_horizon(df, horizon=SETTINGS.horizon)

last_value = (train.sort_values(['unique_id', 'ds'])
                  .groupby('unique_id', observed=True)['y']
                  .last())

naive = (holdout[['unique_id', 'ds']]
         .assign(Naive=lambda d: d['unique_id'].map(last_value).astype(np.float32)))
naive.head()

## Score (bottom-level WRMSSE)

In [ ]:
components = compute_components(train)
scores = wrmsse_for_models(holdout[['unique_id', 'ds', 'y']], naive, components, model_cols=['Naive'])
scores

## Persist for cross-model comparison

In [ ]:
out = SETTINGS.artifacts_dir / 'cv_naive.parquet'

cv_long = (holdout[['unique_id', 'ds', 'y']]
           .assign(cutoff=train['ds'].max(), Naive=naive['Naive'].values))
cv_long.to_parquet(out, index=False)
print(f'wrote {out}')

## Optional — Kaggle-format submission

If you want to score with `datasetsforecast.m5.M5Evaluation` (full 12-level 
WRMSSE) or upload to Kaggle, pivot the long forecast into a wide F1..F28 
frame using `m5.evaluation.make_submission`.

In [ ]:
submission = make_submission(naive, h=SETTINGS.horizon, value_col='Naive', layout='kaggle')
submission.to_parquet(SETTINGS.forecasts_dir / 'forecast_naive.parquet')
submission.head()